In [ ]:

from pathlib import Path
import pandas as pd
import json

# Resolve project root.
# Update PROJECT_ROOT manually only if automatic detection fails.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed").is_dir()
    ),
    Path.cwd(),
)

ROOT = PROJECT_ROOT

print("Project root:", ROOT)
print("Project root exists:", ROOT.exists())
print("Processed data exists:", (ROOT / "data" / "processed").exists())

print("\n=== EXISTING RANDOM FOREST ARTIFACTS ===")

search_roots = [
    ROOT / "data" / "processed" / "classical_ml",
    ROOT / "models",
    ROOT / "reports" / "research_audit",
]

for base in search_roots:
    if not base.exists():
        continue

    for path in sorted(base.rglob("*")):
        if not path.is_file():
            continue

        name = path.name.lower()

        if any(term in name for term in [
            "random_forest",
            "randomforest",
            "classical_ml_model_comparison",
            "final_classical_ml_comparison",
            "classical_ml_predictions",
            "overall_metrics",
            "per_class_metrics",
        ]):
            print(path.relative_to(ROOT))

            if path.suffix.lower() == ".csv":
                try:
                    df = pd.read_csv(path)
                    print("  Shape:", df.shape)
                    print("  Columns:", df.columns.tolist())
                except Exception as exc:
                    print("  Could not read:", repr(exc))

print("\n=== EXISTING CLASSICAL ML DATA FILES ===")

classical_dir = ROOT / "data" / "processed" / "classical_ml"

if classical_dir.exists():
    for path in sorted(classical_dir.iterdir()):
        if path.is_file():
            print(f"{path.name} ({path.stat().st_size:,} bytes)")
else:
    print("Classical ML output directory not found.")


In [ ]:

import joblib
import json
from pathlib import Path

model_paths = {
    "original": ROOT / "models/traditional_ml/random_forest.joblib",
    "patient_disjoint": ROOT / "models/traditional_ml_patient_disjoint/random_forest.joblib",
}

for name, path in model_paths.items():
    print("\n" + "=" * 75)
    print("MODEL:", name)
    print("Path:", path.relative_to(ROOT))
    print("Exists:", path.exists())

    if not path.exists():
        continue

    model = joblib.load(path)
    print("Python class:", type(model).__name__)

    if hasattr(model, "get_params"):
        print("\nHyperparameters:")
        print(json.dumps(model.get_params(), indent=2, default=str))

    if hasattr(model, "n_features_in_"):
        print("\nNumber of input features:", model.n_features_in_)

    if hasattr(model, "feature_names_in_"):
        print("Feature names:")
        print(list(model.feature_names_in_))

    if hasattr(model, "classes_"):
        print("Model classes:")
        print(model.classes_)

print("\n=== SAVED FEATURE SELECTION ===")

feature_path = ROOT / "data/processed/classical_ml/classical_ml_selected_features.csv"
if feature_path.exists():
    import pandas as pd
    features = pd.read_csv(feature_path)
    print("Shape:", features.shape)
    print("Columns:", features.columns.tolist())
    display(features)
else:
    print("Selected-feature file not found.")


In [ ]:

from pathlib import Path
import pandas as pd

print("=== CLASSICAL ML DATA FILES ===")

base = ROOT / "data" / "processed"

for path in sorted(base.rglob("*.csv")):
    rel = str(path.relative_to(ROOT)).lower()

    if any(term in rel for term in [
        "feature", "train", "test", "validation",
        "split", "manifest", "classical_ml"
    ]):
        try:
            df = pd.read_csv(path, nrows=3)
            print(f"\n{path.relative_to(ROOT)}")
            print("Columns:", df.columns.tolist())
        except Exception:
            pass

print("\n=== PATIENT-DISJOINT AUDIT FILES ===")

audit = ROOT / "reports" / "research_audit" / "common_patient_split"

for path in sorted(audit.rglob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["manifest", "split", "config", "feature", "prediction"]
    ):
        print(path.relative_to(ROOT))


In [ ]:

import pandas as pd
import json

feature_path = ROOT / "data/processed/features/respiratory_cycle_features.csv"
manifest_path = ROOT / "data/processed/deep_learning/deep_learning_manifest.csv"
split_path = ROOT / "reports/research_audit/common_patient_split/test_predictions.csv"
selected_path = ROOT / "reports/research_audit/common_patient_split/classical_selected_features.json"

features_df = pd.read_csv(feature_path)
manifest_df = pd.read_csv(manifest_path)
test_predictions_df = pd.read_csv(split_path)

print("=== FEATURE TABLE ===")
print("Shape:", features_df.shape)
print("Columns:", features_df.columns.tolist())
print("\nDiagnosis distribution:")
print(features_df["diagnosis"].value_counts(dropna=False))
print("\nUnique patients:", features_df["patient_id"].nunique())
print("Missing diagnoses:", features_df["diagnosis"].isna().sum())

print("\n=== DEEP-LEARNING MANIFEST ===")
print("Shape:", manifest_df.shape)
print("Columns:", manifest_df.columns.tolist())

for col in ["cnn_split", "official_split", "disease_label", "diagnosis"]:
    if col in manifest_df.columns:
        print(f"\n{col} distribution:")
        print(manifest_df[col].value_counts(dropna=False))

print("\n=== COMMON-PATIENT-SPLIT TEST PREDICTIONS ===")
print("Shape:", test_predictions_df.shape)
print("Columns:", test_predictions_df.columns.tolist())

for col in ["model_id", "patient_id", "recording_id", "true_label",
            "true_disease", "predicted_label", "predicted_disease"]:
    if col in test_predictions_df.columns:
        print(f"\n{col}:")
        print(test_predictions_df[col].value_counts(dropna=False).head(12))

print("\n=== SELECTED FEATURES / AUDIT CONFIGURATION ===")
if selected_path.exists():
    print(selected_path.read_text()[:8000])
else:
    print("Selected-feature JSON not found.")


In [ ]:

import pandas as pd
import numpy as np

audit_dir = ROOT / "reports/research_audit/common_patient_split"
pred_path = audit_dir / "test_predictions.csv"
manifest_path = ROOT / "data/processed/deep_learning/deep_learning_manifest.csv"
features_path = ROOT / "data/processed/features/respiratory_cycle_features.csv"

pred = pd.read_csv(pred_path)
manifest = pd.read_csv(manifest_path)
features = pd.read_csv(features_path)

# Extract the Random Forest rows from the saved common-test predictions.
rf_test = pred.loc[pred["model_id"].eq("random_forest")].copy()

print("=== COMMON TEST SET ===")
print("RF prediction rows:", len(rf_test))
print("Unique test patients:", rf_test["patient_id"].nunique())
print("Unique test recordings:", rf_test["recording_id"].nunique())
print("True-label distribution:")
print(rf_test["true_disease"].value_counts())

print("\n=== MANIFEST SPLIT INFORMATION ===")
print("Manifest columns:", manifest.columns.tolist())

if "cnn_split" in manifest.columns:
    print(manifest["cnn_split"].value_counts(dropna=False))
else:
    print("No cnn_split column in the manifest.")

print("\n=== PATIENT OVERLAP: OFFICIAL TRAIN VS COMMON TEST ===")
official_train = set(
    manifest.loc[manifest["official_split"].eq("train"), "patient_id"].dropna()
)
common_test_patients = set(rf_test["patient_id"].dropna())

print("Official training patients:", len(official_train))
print("Common test patients:", len(common_test_patients))
print("Overlap:", len(official_train & common_test_patients))
print("Common test patients absent from official training:",
      len(common_test_patients - official_train))

print("\n=== TEST LABEL CONSISTENCY ===")
feature_labels = (
    features[["patient_id", "recording_id", "diagnosis"]]
    .drop_duplicates()
)

check = rf_test.merge(
    feature_labels,
    on=["patient_id", "recording_id"],
    how="left",
    suffixes=("_prediction", "_feature")
)

print("Rows after merge:", len(check))
print("Rows without matching feature record:", check["diagnosis"].isna().sum())

if "true_disease" in check.columns:
    matched = check["diagnosis"].eq(check["true_disease"])
    print("Diagnosis matches:", int(matched.sum()), "/", len(check))
    print("Diagnosis mismatch rows:", int((~matched).sum()))

print("\n=== SAVED SPLIT / CONFIGURATION FILES ===")
for path in sorted(audit_dir.rglob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["manifest", "patient", "split", "config"]
    ):
        print(path.relative_to(ROOT))


In [ ]:


# ============================================================
# CELL 1: DEFINE PROJECT ROOT DIRECTORY
# ============================================================

from pathlib import Path
import os

# Search common locations for the project directory
possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / "Sem 5" / "Respiratory-Disease-Classification",
    Path.home() / "Respiratory-Disease-Classification",
]

required_file = Path(
    "data/processed/features/respiratory_cycle_features.csv"
)

ROOT = None

for candidate in possible_roots:
    if (candidate / required_file).exists():
        ROOT = candidate.resolve()
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not automatically locate the project root.\n"
        f"Current working directory: {Path.cwd()}\n"
        "Please set ROOT manually to your "
        "Respiratory-Disease-Classification project directory."
    )

print("Project root:", ROOT)
print("Feature file exists:", (ROOT / required_file).exists())

print("\nProject root contents:")
for item in sorted(ROOT.iterdir()):
    print(" -", item.name)


In [ ]:
# ============================================================
# CELL 7: PATIENT-LEVEL DATA AUDIT BEFORE MODEL OPTIMIZATION
# ============================================================

import pandas as pd
import numpy as np
import json

from pathlib import Path

print("=" * 80)
print("RANDOM FOREST OPTIMIZATION: PATIENT-LEVEL DATA AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load the original feature dataset and manifest
# ------------------------------------------------------------

feature_path = (
    ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

manifest_path = (
    ROOT / "data" / "processed" / "deep_learning"
    / "deep_learning_manifest.csv"
)

features_df = pd.read_csv(feature_path)
manifest_df = pd.read_csv(manifest_path)

required_columns = {
    "patient_id",
    "diagnosis",
}

missing = required_columns - set(features_df.columns)

if missing:
    raise ValueError(
        f"Missing required columns in feature dataset: {missing}"
    )

print("\n[1] DATASET SUMMARY")
print("-" * 80)

print("Feature dataset shape:", features_df.shape)
print("Unique patients:", features_df["patient_id"].nunique())
print("Unique diagnoses:", features_df["diagnosis"].nunique())
print("Missing patient IDs:", features_df["patient_id"].isna().sum())
print("Missing diagnoses:", features_df["diagnosis"].isna().sum())

if "recording_id" in features_df.columns:
    print(
        "Unique recordings:",
        features_df[["patient_id", "recording_id"]]
        .drop_duplicates()
        .shape[0]
    )

# ------------------------------------------------------------
# 2. Disease support: cycles, recordings and patients
# ------------------------------------------------------------

print("\n[2] DISEASE-LEVEL SUPPORT")
print("-" * 80)

support = (
    features_df.groupby("diagnosis", dropna=False)
    .agg(
        cycles=("diagnosis", "size"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
    .rename(columns={"diagnosis": "disease"})
)

if "recording_id" in features_df.columns:
    recording_counts = (
        features_df.groupby("diagnosis")
        .apply(
            lambda g: g[["patient_id", "recording_id"]]
            .drop_duplicates()
            .shape[0],
            include_groups=False,
        )
        .rename("recordings")
        .reset_index()
        .rename(columns={"diagnosis": "disease"})
    )

    support = support.merge(
        recording_counts,
        on="disease",
        how="left",
    )

support["cycle_percentage"] = (
    support["cycles"] / len(features_df) * 100
).round(2)

support = support.sort_values("cycles", ascending=False)

display(support)

# ------------------------------------------------------------
# 3. Check whether patient IDs have inconsistent diagnoses
# ------------------------------------------------------------

print("\n[3] PATIENT DIAGNOSIS CONSISTENCY")
print("-" * 80)

diagnoses_per_patient = (
    features_df.groupby("patient_id")["diagnosis"]
    .nunique()
)

multi_diagnosis_patients = diagnoses_per_patient[
    diagnoses_per_patient > 1
]

print(
    "Patients associated with more than one diagnosis:",
    len(multi_diagnosis_patients),
)

if len(multi_diagnosis_patients):
    print("\nExamples requiring investigation:")
    display(
        multi_diagnosis_patients
        .rename("number_of_diagnoses")
        .reset_index()
        .head(20)
    )
else:
    print("Each patient has one recorded diagnosis.")

# ------------------------------------------------------------
# 4. Inspect patient overlap in the official split
# ------------------------------------------------------------

print("\n[4] OFFICIAL SPLIT INTEGRITY")
print("-" * 80)

if {
    "patient_id",
    "official_split",
}.issubset(manifest_df.columns):

    official_train_patients = set(
        manifest_df.loc[
            manifest_df["official_split"].eq("train"),
            "patient_id",
        ].dropna()
    )

    official_test_patients = set(
        manifest_df.loc[
            manifest_df["official_split"].eq("test"),
            "patient_id",
        ].dropna()
    )

    overlap = official_train_patients & official_test_patients

    print("Official training patients:", len(official_train_patients))
    print("Official test patients:", len(official_test_patients))
    print("Patient overlap:", len(overlap))

    if overlap:
        print(
            "\nWARNING: The official split contains patient overlap."
        )
        print("Overlapping patient IDs:", sorted(overlap))
    else:
        print("Official train and test patients are disjoint.")

else:
    print("Required official split columns were not found.")

# ------------------------------------------------------------
# 5. Compare official test patients with the common-test audit
# ------------------------------------------------------------

print("\n[5] COMMON-TEST PATIENT OVERLAP")
print("-" * 80)

common_predictions_path = (
    ROOT / "reports" / "research_audit"
    / "common_patient_split" / "test_predictions.csv"
)

if common_predictions_path.exists():

    predictions_df = pd.read_csv(common_predictions_path)

    if "model_id" in predictions_df.columns:
        rf_predictions = predictions_df.loc[
            predictions_df["model_id"].eq("random_forest")
        ].copy()
    else:
        rf_predictions = predictions_df.copy()

    if "patient_id" in rf_predictions.columns:

        common_test_patients = set(
            rf_predictions["patient_id"].dropna()
        )

        official_train_overlap = (
            common_test_patients & official_train_patients
            if "official_train_patients" in locals()
            else set()
        )

        print(
            "Common-test patients:",
            len(common_test_patients),
        )

        print(
            "Common-test patients also in official training:",
            len(official_train_overlap),
        )

        if official_train_overlap:
            print(
                "Overlapping patients:",
                sorted(official_train_overlap),
            )

            print(
                "\nWARNING: Do not use this common-test result as "
                "a fully patient-independent benchmark for a model "
                "trained on the official training split."
            )

else:
    print("Common-test prediction file not found.")

# ------------------------------------------------------------
# 6. Check feature quality
# ------------------------------------------------------------

print("\n[6] NUMERICAL FEATURE QUALITY")
print("-" * 80)

excluded_columns = {
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
}

numeric_feature_columns = [
    c for c in features_df.select_dtypes(include=np.number).columns
    if c not in excluded_columns
]

X_check = features_df[numeric_feature_columns].replace(
    [np.inf, -np.inf],
    np.nan,
)

feature_quality = pd.DataFrame({
    "feature": numeric_feature_columns,
    "missing_values": X_check.isna().sum().values,
    "infinite_or_missing_after_cleanup": (
        features_df[numeric_feature_columns]
        .replace([np.inf, -np.inf], np.nan)
        .isna()
        .sum()
        .values
    ),
    "unique_values": [
        features_df[c].nunique(dropna=True)
        for c in numeric_feature_columns
    ],
})

feature_quality["constant_feature"] = (
    feature_quality["unique_values"] <= 1
)

print("Numerical features found:", len(numeric_feature_columns))
print("Features with missing/non-finite values:",
      int((feature_quality["infinite_or_missing_after_cleanup"] > 0).sum()))
print("Constant features:",
      int(feature_quality["constant_feature"].sum()))

display(
    feature_quality.sort_values(
        ["constant_feature", "infinite_or_missing_after_cleanup"],
        ascending=False,
    )
)

# ------------------------------------------------------------
# 7. Save audit results for reproducibility
# ------------------------------------------------------------

audit_output_dir = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

audit_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

support.to_csv(
    audit_output_dir / "disease_patient_support.csv",
    index=False,
)

feature_quality.to_csv(
    audit_output_dir / "feature_quality_audit.csv",
    index=False,
)

audit_summary = {
    "feature_dataset_rows": int(len(features_df)),
    "unique_patients": int(features_df["patient_id"].nunique()),
    "unique_diseases": int(features_df["diagnosis"].nunique()),
    "numeric_features": int(len(numeric_feature_columns)),
    "patients_with_multiple_diagnoses": int(
        len(multi_diagnosis_patients)
    ),
    "disease_support": support.to_dict(orient="records"),
}

with open(
    audit_output_dir / "dataset_audit_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(audit_summary, f, indent=2, default=str)

print("\n[7] SAVED AUDIT OUTPUTS")
print("-" * 80)

print("Output directory:", audit_output_dir)
print(" - disease_patient_support.csv")
print(" - feature_quality_audit.csv")
print(" - dataset_audit_summary.json")

print("\nAUDIT COMPLETE.")
print(
    "Next step: use the patient-support results to determine "
    "a feasible grouped cross-validation strategy."
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# ============================================================
# RANDOM FOREST: PATIENT-GROUPED CROSS-VALIDATION AUDIT
# ============================================================

print("=" * 90)
print("PATIENT-GROUPED CROSS-VALIDATION FEASIBILITY AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. FIND PROJECT ROOT
# ------------------------------------------------------------

if "ROOT" not in globals():
    possible_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Sem 5" / "Respiratory-Disease-Classification",
        Path.home() / "Respiratory-Disease-Classification",
    ]

    ROOT = next(
        (
            p.resolve()
            for p in possible_roots
            if (
                p / "data/processed/features/"
                "respiratory_cycle_features.csv"
            ).exists()
        ),
        None,
    )

if ROOT is None or not (
    ROOT / "data/processed/features/"
    "respiratory_cycle_features.csv"
).exists():
    raise FileNotFoundError(
        "Could not locate respiratory_cycle_features.csv. "
        "Set ROOT manually to your project directory."
    )

print("\nProject root:", ROOT)

# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

FEATURE_PATH = (
    ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

df = pd.read_csv(FEATURE_PATH)

print("\nDataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

# ------------------------------------------------------------
# 3. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = ["patient_id", "diagnosis"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        "Check the dataset column names and update the code."
    )

if df["patient_id"].isna().any():
    raise ValueError("Missing patient IDs found.")

if df["diagnosis"].isna().any():
    raise ValueError("Missing diagnosis labels found.")

# ------------------------------------------------------------
# 4. CHECK PATIENT DIAGNOSIS CONSISTENCY
# ------------------------------------------------------------

diagnoses_per_patient = (
    df.groupby("patient_id")["diagnosis"].nunique()
)

inconsistent_patients = diagnoses_per_patient[
    diagnoses_per_patient > 1
]

print(
    "\nPatients with multiple diagnoses:",
    len(inconsistent_patients)
)

if len(inconsistent_patients) > 0:
    print(inconsistent_patients)
    raise ValueError(
        "Resolve inconsistent patient diagnoses before CV."
    )

# ------------------------------------------------------------
# 5. DISEASE-LEVEL PATIENT SUPPORT
# ------------------------------------------------------------

patient_support = (
    df.groupby("diagnosis")
    .agg(
        cycles=("patient_id", "size"),
        patients=("patient_id", "nunique"),
    )
    .sort_values("patients", ascending=True)
)

print("\n" + "=" * 90)
print("DISEASE-LEVEL SUPPORT")
print("=" * 90)
print(patient_support.to_string())

# ------------------------------------------------------------
# 6. CREATE FIVE-FOLD PATIENT-GROUPED SPLITS
# ------------------------------------------------------------

y = df["diagnosis"].astype(str).to_numpy()
groups = df["patient_id"].to_numpy()

# Dummy X is sufficient because this step only constructs splits.
X_dummy = np.zeros((len(df), 1))

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold_splits = list(cv.split(X_dummy, y, groups))

fold_rows = []
fold_class_support = []

for fold_number, (train_idx, val_idx) in enumerate(
    fold_splits, start=1
):
    train_patients = set(groups[train_idx])
    val_patients = set(groups[val_idx])

    patient_overlap = train_patients.intersection(val_patients)

    train_classes = set(y[train_idx])
    val_classes = set(y[val_idx])

    print("\n" + "=" * 90)
    print(f"FOLD {fold_number}")
    print("=" * 90)

    print("Training cycles:", len(train_idx))
    print("Validation cycles:", len(val_idx))
    print("Training patients:", len(train_patients))
    print("Validation patients:", len(val_patients))
    print("Patient overlap:", len(patient_overlap))
    print("Training classes:", len(train_classes))
    print("Validation classes:", len(val_classes))

    if patient_overlap:
        raise AssertionError(
            f"Patient leakage detected in fold {fold_number}"
        )

    train_counts = (
        df.iloc[train_idx]["diagnosis"]
        .value_counts()
        .sort_index()
    )

    val_counts = (
        df.iloc[val_idx]["diagnosis"]
        .value_counts()
        .sort_index()
    )

    class_table = pd.concat(
        [
            train_counts.rename("train_cycles"),
            val_counts.rename("validation_cycles"),
        ],
        axis=1,
    ).fillna(0).astype(int)

    class_table["fold"] = fold_number
    class_table.index.name = "diagnosis"

    fold_class_support.append(class_table.reset_index())

    print("\nValidation disease support:")
    print(val_counts.to_string())

    missing_validation_classes = sorted(
        train_classes - val_classes
    )

    missing_training_classes = sorted(
        val_classes - train_classes
    )

    print(
        "\nClasses present in training but absent "
        "from validation:",
        missing_validation_classes,
    )

    print(
        "Classes present in validation but absent "
        "from training:",
        missing_training_classes,
    )

    fold_rows.append(
        {
            "fold": fold_number,
            "train_cycles": len(train_idx),
            "validation_cycles": len(val_idx),
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients),
            "patient_overlap": len(patient_overlap),
            "train_classes": len(train_classes),
            "validation_classes": len(val_classes),
            "classes_missing_from_validation": ", ".join(
                missing_validation_classes
            ),
            "classes_missing_from_training": ", ".join(
                missing_training_classes
            ),
        }
    )

# ------------------------------------------------------------
# 7. SUMMARIZE FOLD FEASIBILITY
# ------------------------------------------------------------

fold_report = pd.DataFrame(fold_rows)

print("\n" + "=" * 90)
print("FIVE-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 90)

print(fold_report.to_string(index=False))

# ------------------------------------------------------------
# 8. SAVE AUDIT REPORTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patient_support.to_csv(
    OUTPUT_DIR / "patient_support_by_disease.csv"
)

fold_report.to_csv(
    OUTPUT_DIR / "grouped_cv_feasibility.csv",
    index=False,
)

pd.concat(
    fold_class_support,
    ignore_index=True,
).to_csv(
    OUTPUT_DIR / "grouped_cv_class_support.csv",
    index=False,
)

print("\n" + "=" * 90)
print("AUDIT COMPLETE")
print("=" * 90)

print("Reports saved to:", OUTPUT_DIR)
print("- patient_support_by_disease.csv")
print("- grouped_cv_feasibility.csv")
print("- grouped_cv_class_support.csv")

print("\nIMPORTANT:")
print("1. These are candidate folds, not model performance results.")
print("2. Patient overlap is checked explicitly.")
print("3. Rare diseases may be absent from some validation folds.")
print("4. Five-fold feasibility does not guarantee every class")
print("   appears in every training and validation fold.")
print("5. No Random Forest hyperparameter tuning has been performed.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# ============================================================
# FIVE-CLASS RANDOM FOREST: PATIENT-GROUPED CV PREPARATION
# ============================================================

print("=" * 80)
print("FIVE-CLASS PATIENT-GROUPED CROSS-VALIDATION")
print("=" * 80)

# 1. FIND PROJECT ROOT

if "ROOT" not in globals():
    possible_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "Sem 5" / "Respiratory-Disease-Classification",
        Path.home() / "Respiratory-Disease-Classification",
    ]

    ROOT = next(
        (
            p.resolve()
            for p in possible_roots
            if (
                p / "data/processed/features/"
                "respiratory_cycle_features.csv"
            ).exists()
        ),
        None,
    )

if ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Set ROOT manually."
    )

# 2. LOAD DATASET

FEATURE_PATH = (
    ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(FEATURE_PATH)

df = pd.read_csv(FEATURE_PATH)

required_columns = {"patient_id", "diagnosis"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        f"Missing columns: {required_columns - set(df.columns)}"
    )

if df["patient_id"].isna().any() or df["diagnosis"].isna().any():
    raise ValueError("Missing patient IDs or diagnoses detected.")

# 3. SELECT FIVE CLASSES

TARGET_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

df5 = df[df["diagnosis"].isin(TARGET_CLASSES)].copy()
df5.reset_index(drop=True, inplace=True)

print("\nProject root:", ROOT)
print("Original dataset shape:", df.shape)
print("Five-class dataset shape:", df5.shape)
print("Unique patients:", df5["patient_id"].nunique())

# 4. CHECK PATIENT-LEVEL CLASS SUPPORT

support = (
    df5.groupby("diagnosis")
    .agg(
        cycles=("patient_id", "size"),
        patients=("patient_id", "nunique"),
    )
    .reindex(TARGET_CLASSES)
)

print("\nCLASS SUPPORT")
print(support.to_string())

if support.isna().any().any():
    raise ValueError("One or more target classes are missing.")

# 5. CHECK PATIENT DIAGNOSIS CONSISTENCY

diagnoses_per_patient = (
    df5.groupby("patient_id")["diagnosis"].nunique()
)

if (diagnoses_per_patient > 1).any():
    raise ValueError(
        "Some patients have multiple diagnoses."
    )

# 6. GENERATE FIVE GROUPED FOLDS

y = df5["diagnosis"].astype(str).to_numpy()
groups = df5["patient_id"].to_numpy()
X_dummy = np.zeros((len(df5), 1))

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold_assignments = np.full(len(df5), -1, dtype=int)
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_dummy, y, groups),
    start=1,
):
    train_patients = set(groups[train_idx])
    val_patients = set(groups[val_idx])

    overlap = train_patients.intersection(val_patients)

    train_classes = set(y[train_idx])
    val_classes = set(y[val_idx])

    missing_train = sorted(
        set(TARGET_CLASSES) - train_classes
    )
    missing_val = sorted(
        set(TARGET_CLASSES) - val_classes
    )

    train_support = (
        df5.iloc[train_idx]["diagnosis"]
        .value_counts()
        .reindex(TARGET_CLASSES, fill_value=0)
    )

    val_support = (
        df5.iloc[val_idx]["diagnosis"]
        .value_counts()
        .reindex(TARGET_CLASSES, fill_value=0)
    )

    print(f"\n{'=' * 60}")
    print(f"FOLD {fold}")
    print("=" * 60)

    print("Training cycles:", len(train_idx))
    print("Validation cycles:", len(val_idx))
    print("Training patients:", len(train_patients))
    print("Validation patients:", len(val_patients))
    print("Patient overlap:", len(overlap))

    print("\nTraining cycles by class:")
    print(train_support.to_string())

    print("\nValidation cycles by class:")
    print(val_support.to_string())

    print("\nMissing training classes:", missing_train)
    print("Missing validation classes:", missing_val)

    if overlap:
        raise AssertionError(
            f"Patient leakage detected in fold {fold}."
        )

    if missing_train or missing_val:
        raise ValueError(
            f"Fold {fold} lacks one or more classes. "
            "Review the grouped cross-validation strategy."
        )

    fold_assignments[val_idx] = fold

    fold_rows.append({
        "fold": fold,
        "train_cycles": len(train_idx),
        "validation_cycles": len(val_idx),
        "train_patients": len(train_patients),
        "validation_patients": len(val_patients),
        "patient_overlap": len(overlap),
        **{
            f"validation_{cls}_cycles": int(val_support[cls])
            for cls in TARGET_CLASSES
        },
    })

# 7. VERIFY ALL ROWS WERE ASSIGNED

if np.any(fold_assignments == -1):
    raise AssertionError("Some rows were not assigned to a fold.")

df5["cv_fold"] = fold_assignments

# 8. SAVE REPORTS

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assignments_path = (
    OUTPUT_DIR / "five_class_grouped_cv_assignments.csv"
)

summary_path = (
    OUTPUT_DIR / "five_class_grouped_cv_summary.csv"
)

df5.to_csv(assignments_path, index=False)

pd.DataFrame(fold_rows).to_csv(
    summary_path,
    index=False,
)

print("\n" + "=" * 80)
print("CROSS-VALIDATION PREPARATION COMPLETE")
print("=" * 80)

print("\nFold summary:")
print(pd.DataFrame(fold_rows).to_string(index=False))

print("\nSaved files:")
print(assignments_path)
print(summary_path)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold

# ============================================================
# RANDOM FOREST BASELINE: FIVE-CLASS GROUPED CROSS-VALIDATION
# ============================================================

print("=" * 90)
print("RANDOM FOREST BASELINE - PATIENT-GROUPED 5-FOLD CV")
print("=" * 90)

# ------------------------------------------------------------
# 1. LOAD DATA AND PREPARED FOLD ASSIGNMENTS
# ------------------------------------------------------------

FEATURE_PATH = (
    ROOT / "data" / "processed" / "features"
    / "respiratory_cycle_features.csv"
)

ASSIGNMENT_PATH = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
    / "five_class_grouped_cv_assignments.csv"
)

if not FEATURE_PATH.exists():
    raise FileNotFoundError(FEATURE_PATH)

if not ASSIGNMENT_PATH.exists():
    raise FileNotFoundError(ASSIGNMENT_PATH)

df = pd.read_csv(FEATURE_PATH)
df5 = pd.read_csv(ASSIGNMENT_PATH)

TARGET_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

print("\nFive-class dataset:", df5.shape)
print("Patients:", df5["patient_id"].nunique())

# ------------------------------------------------------------
# 2. SELECT NUMERICAL FEATURES
# ------------------------------------------------------------

EXCLUDED_COLUMNS = {
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
    "cv_fold",
}

feature_columns = [
    col
    for col in df5.select_dtypes(include=[np.number]).columns
    if col not in EXCLUDED_COLUMNS
]

if not feature_columns:
    raise ValueError("No numerical input features found.")

X = df5[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)

if X.isna().any().any():
    bad_features = X.columns[X.isna().any()].tolist()
    raise ValueError(
        f"Missing or infinite values in features: {bad_features}"
    )

y = df5["diagnosis"].astype(str).to_numpy()
groups = df5["patient_id"].to_numpy()
fold_ids = df5["cv_fold"].to_numpy()

print("\nNumber of numerical features:", len(feature_columns))
print("Features:", feature_columns)

# ------------------------------------------------------------
# 3. CONFIGURE BASELINE RANDOM FOREST
# ------------------------------------------------------------

BASELINE_PARAMS = {
    "n_estimators": 300,
    "random_state": 42,
    "n_jobs": -1,
    "class_weight": "balanced",
}

print("\nBaseline parameters:")
print(BASELINE_PARAMS)

# ------------------------------------------------------------
# 4. TRAIN AND EVALUATE EACH FOLD
# ------------------------------------------------------------

oof_predictions = np.empty(len(df5), dtype=object)
oof_probabilities = np.full(
    (len(df5), len(TARGET_CLASSES)),
    np.nan,
    dtype=float,
)

fold_results = []
fold_reports = []
confusion_matrices = {}

for fold in range(1, 6):

    train_idx = np.flatnonzero(fold_ids != fold)
    val_idx = np.flatnonzero(fold_ids == fold)

    train_patients = set(groups[train_idx])
    val_patients = set(groups[val_idx])

    overlap = train_patients.intersection(val_patients)

    if overlap:
        raise AssertionError(
            f"Patient leakage detected in fold {fold}: {overlap}"
        )

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]

    model = RandomForestClassifier(**BASELINE_PARAMS)

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)

    model_classes = list(model.classes_)

    for j, class_name in enumerate(model_classes):
        class_position = TARGET_CLASSES.index(class_name)
        oof_probabilities[val_idx, class_position] = probabilities[:, j]

    oof_predictions[val_idx] = predictions

    accuracy = accuracy_score(y_val, predictions)

    balanced_accuracy = balanced_accuracy_score(
        y_val, predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        labels=TARGET_CLASSES,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        y_val,
        predictions,
        labels=TARGET_CLASSES,
        average="weighted",
        zero_division=0,
    )

    report = classification_report(
        y_val,
        predictions,
        labels=TARGET_CLASSES,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_val,
        predictions,
        labels=TARGET_CLASSES,
    )

    confusion_matrices[fold] = cm

    fold_results.append({
        "fold": fold,
        "train_cycles": len(train_idx),
        "validation_cycles": len(val_idx),
        "train_patients": len(train_patients),
        "validation_patients": len(val_patients),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    })

    for class_name in TARGET_CLASSES:
        fold_reports.append({
            "fold": fold,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1_score": report[class_name]["f1-score"],
            "support": report[class_name]["support"],
        })

    print(f"\n{'=' * 70}")
    print(f"FOLD {fold}")
    print("=" * 70)
    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_accuracy:.4f}")
    print(f"Macro-F1:          {macro_f1:.4f}")
    print(f"Weighted-F1:       {weighted_f1:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_val,
            predictions,
            labels=TARGET_CLASSES,
            zero_division=0,
        )
    )

# ------------------------------------------------------------
# 5. COMBINED OUT-OF-FOLD METRICS
# ------------------------------------------------------------

if pd.isna(pd.Series(oof_predictions)).any():
    raise AssertionError("Some rows have no out-of-fold prediction.")

if np.isnan(oof_probabilities).any():
    raise AssertionError("Some rows have no out-of-fold probability.")

overall_accuracy = accuracy_score(y, oof_predictions)

overall_balanced_accuracy = balanced_accuracy_score(
    y, oof_predictions
)

overall_macro_f1 = f1_score(
    y,
    oof_predictions,
    labels=TARGET_CLASSES,
    average="macro",
    zero_division=0,
)

overall_weighted_f1 = f1_score(
    y,
    oof_predictions,
    labels=TARGET_CLASSES,
    average="weighted",
    zero_division=0,
)

print("\n" + "=" * 90)
print("COMBINED OUT-OF-FOLD RESULTS")
print("=" * 90)

print(f"Accuracy:          {overall_accuracy:.4f}")
print(f"Balanced accuracy: {overall_balanced_accuracy:.4f}")
print(f"Macro-F1:          {overall_macro_f1:.4f}")
print(f"Weighted-F1:       {overall_weighted_f1:.4f}")

print("\nCombined classification report:")
print(
    classification_report(
        y,
        oof_predictions,
        labels=TARGET_CLASSES,
        zero_division=0,
    )
)

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fold_results_df = pd.DataFrame(fold_results)
fold_reports_df = pd.DataFrame(fold_reports)

fold_results_df.to_csv(
    OUTPUT_DIR / "rf_baseline_fold_metrics.csv",
    index=False,
)

fold_reports_df.to_csv(
    OUTPUT_DIR / "rf_baseline_per_class_metrics.csv",
    index=False,
)

oof_df = df5[
    ["patient_id", "recording_id", "diagnosis", "cv_fold"]
].copy()

oof_df["prediction"] = oof_predictions

for j, class_name in enumerate(TARGET_CLASSES):
    oof_df[f"probability_{class_name}"] = oof_probabilities[:, j]

oof_df.to_csv(
    OUTPUT_DIR / "rf_baseline_oof_predictions.csv",
    index=False,
)

for fold, cm in confusion_matrices.items():
    pd.DataFrame(
        cm,
        index=TARGET_CLASSES,
        columns=TARGET_CLASSES,
    ).to_csv(
        OUTPUT_DIR / f"rf_baseline_confusion_matrix_fold_{fold}.csv"
    )

summary = {
    "experiment": "five_class_patient_grouped_rf_baseline",
    "classes": TARGET_CLASSES,
    "n_cycles": int(len(df5)),
    "n_patients": int(df5["patient_id"].nunique()),
    "n_features": int(len(feature_columns)),
    "model_parameters": BASELINE_PARAMS,
    "oof_accuracy": float(overall_accuracy),
    "oof_balanced_accuracy": float(overall_balanced_accuracy),
    "oof_macro_f1": float(overall_macro_f1),
    "oof_weighted_f1": float(overall_weighted_f1),
    "fold_macro_f1_mean": float(fold_results_df["macro_f1"].mean()),
    "fold_macro_f1_std": float(fold_results_df["macro_f1"].std()),
    "fold_balanced_accuracy_mean": float(
        fold_results_df["balanced_accuracy"].mean()
    ),
    "fold_balanced_accuracy_std": float(
        fold_results_df["balanced_accuracy"].std()
    ),
}

with open(
    OUTPUT_DIR / "rf_baseline_summary.json",
    "w",
) as f:
    json.dump(summary, f, indent=4)

print("\nSaved baseline results to:")
print(OUTPUT_DIR)

print("\nFiles created:")
print("- rf_baseline_fold_metrics.csv")
print("- rf_baseline_per_class_metrics.csv")
print("- rf_baseline_oof_predictions.csv")
print("- rf_baseline_confusion_matrix_fold_1.csv through fold_5.csv")
print("- rf_baseline_summary.json")

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# ============================================================
# RANDOM FOREST: CLASS-IMBALANCE STRATEGY COMPARISON
# ============================================================

print("=" * 90)
print("RANDOM FOREST CLASS-IMBALANCE EXPERIMENTS")
print("=" * 90)

# ------------------------------------------------------------
# 1. LOAD THE PREPARED DATA AND FOLDS
# ------------------------------------------------------------

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

ASSIGNMENT_PATH = (
    OUTPUT_DIR / "five_class_grouped_cv_assignments.csv"
)

df5 = pd.read_csv(ASSIGNMENT_PATH)

TARGET_CLASSES = [
    "COPD",
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

EXCLUDED_COLUMNS = {
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
    "cv_fold",
}

feature_columns = [
    col
    for col in df5.select_dtypes(include=[np.number]).columns
    if col not in EXCLUDED_COLUMNS
]

X = df5[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)

if X.isna().any().any():
    raise ValueError("Features contain missing or infinite values.")

y = df5["diagnosis"].astype(str).to_numpy()
groups = df5["patient_id"].to_numpy()
fold_ids = df5["cv_fold"].to_numpy()

print("Dataset shape:", df5.shape)
print("Patients:", df5["patient_id"].nunique())
print("Features:", len(feature_columns))

# ------------------------------------------------------------
# 2. DEFINE EXPERIMENTS
# ------------------------------------------------------------

experiments = [
    {
        "name": "RF_balanced",
        "class_weight": "balanced",
    },
    {
        "name": "RF_balanced_subsample",
        "class_weight": "balanced_subsample",
    },
    {
        "name": "RF_sqrt_inverse_frequency",
        "class_weight": None,
    },
]

# ------------------------------------------------------------
# 3. OPTIONAL BALANCED RANDOM FOREST
# ------------------------------------------------------------

try:
    from imblearn.ensemble import BalancedRandomForestClassifier

    experiments.append({
        "name": "BalancedRandomForest",
        "class_weight": None,
        "balanced_rf": True,
    })

    print("\nimbalanced-learn is installed.")
    print("BalancedRandomForestClassifier will be evaluated.")

except ImportError:
    print("\nimbalanced-learn is not installed.")
    print("Skipping BalancedRandomForestClassifier.")

# ------------------------------------------------------------
# 4. RUN ALL EXPERIMENTS ON IDENTICAL FOLDS
# ------------------------------------------------------------

fold_results = []
class_results = []
oof_predictions = {}

for experiment in experiments:

    experiment_name = experiment["name"]

    print("\n" + "=" * 90)
    print("EXPERIMENT:", experiment_name)
    print("=" * 90)

    predictions_all = np.empty(len(df5), dtype=object)
    predictions_all[:] = None

    start_time = time.time()

    for fold in range(1, 6):

        train_idx = np.flatnonzero(fold_ids != fold)
        val_idx = np.flatnonzero(fold_ids == fold)

        train_patients = set(groups[train_idx])
        val_patients = set(groups[val_idx])

        if train_patients.intersection(val_patients):
            raise AssertionError(
                f"Patient leakage in fold {fold}"
            )

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        if experiment.get("balanced_rf", False):

            model = BalancedRandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                sampling_strategy="all",
                replacement=True,
            )

        elif experiment_name == "RF_sqrt_inverse_frequency":

            classes = np.unique(y_train)

            class_weights = compute_class_weight(
                class_weight="balanced",
                classes=classes,
                y=y_train,
            )

            # Square-root weights are less aggressive than
            # full inverse-frequency weighting.
            class_weights = np.sqrt(class_weights)

            weight_map = dict(zip(classes, class_weights))

            model = RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight=weight_map,
            )

        else:

            model = RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
                class_weight=experiment["class_weight"],
            )

        model.fit(X_train, y_train)

        predictions = model.predict(X_val)
        predictions_all[val_idx] = predictions

        accuracy = accuracy_score(y_val, predictions)

        balanced_accuracy = balanced_accuracy_score(
            y_val, predictions
        )

        macro_f1 = f1_score(
            y_val,
            predictions,
            labels=TARGET_CLASSES,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_val,
            predictions,
            labels=TARGET_CLASSES,
            average="weighted",
            zero_division=0,
        )

        fold_results.append({
            "experiment": experiment_name,
            "fold": fold,
            "accuracy": accuracy,
            "balanced_accuracy": balanced_accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "train_patients": len(train_patients),
            "validation_patients": len(val_patients),
        })

        precision, recall, f1_values, supports = (
            precision_recall_fscore_support(
                y_val,
                predictions,
                labels=TARGET_CLASSES,
                zero_division=0,
            )
        )

        for i, class_name in enumerate(TARGET_CLASSES):
            class_results.append({
                "experiment": experiment_name,
                "fold": fold,
                "class": class_name,
                "precision": precision[i],
                "recall": recall[i],
                "f1_score": f1_values[i],
                "support": supports[i],
            })

        print(
            f"Fold {fold}: "
            f"Accuracy={accuracy:.4f}, "
            f"Balanced Accuracy={balanced_accuracy:.4f}, "
            f"Macro-F1={macro_f1:.4f}"
        )

    if any(pred is None for pred in predictions_all):
        raise AssertionError(
            f"Incomplete predictions for {experiment_name}"
        )

    elapsed = time.time() - start_time

    oof_predictions[experiment_name] = predictions_all

    print(f"Runtime: {elapsed:.1f} seconds")

# ------------------------------------------------------------
# 5. SUMMARIZE RESULTS
# ------------------------------------------------------------

fold_df = pd.DataFrame(fold_results)
class_df = pd.DataFrame(class_results)

summary_rows = []

for experiment_name, group in fold_df.groupby("experiment"):

    predictions = oof_predictions[experiment_name]

    summary_rows.append({
        "experiment": experiment_name,
        "oof_accuracy": accuracy_score(y, predictions),
        "oof_balanced_accuracy": balanced_accuracy_score(
            y, predictions
        ),
        "oof_macro_f1": f1_score(
            y,
            predictions,
            labels=TARGET_CLASSES,
            average="macro",
            zero_division=0,
        ),
        "oof_weighted_f1": f1_score(
            y,
            predictions,
            labels=TARGET_CLASSES,
            average="weighted",
            zero_division=0,
        ),
        "mean_fold_macro_f1": group["macro_f1"].mean(),
        "std_fold_macro_f1": group["macro_f1"].std(),
        "mean_fold_balanced_accuracy": group[
            "balanced_accuracy"
        ].mean(),
        "std_fold_balanced_accuracy": group[
            "balanced_accuracy"
        ].std(),
    })

summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df.sort_values(
    "oof_macro_f1",
    ascending=False,
)

print("\n" + "=" * 90)
print("IMBALANCE STRATEGY COMPARISON")
print("=" * 90)

print(summary_df.to_string(index=False))

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

fold_df.to_csv(
    OUTPUT_DIR / "rf_imbalance_fold_metrics.csv",
    index=False,
)

class_df.to_csv(
    OUTPUT_DIR / "rf_imbalance_per_class_metrics.csv",
    index=False,
)

summary_df.to_csv(
    OUTPUT_DIR / "rf_imbalance_summary.csv",
    index=False,
)

oof_df = df5[
    ["patient_id", "recording_id", "diagnosis", "cv_fold"]
].copy()

for experiment_name, predictions in oof_predictions.items():
    oof_df[f"prediction_{experiment_name}"] = predictions

oof_df.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_predictions.csv",
    index=False,
)

print("\nSaved files:")
print(OUTPUT_DIR / "rf_imbalance_fold_metrics.csv")
print(OUTPUT_DIR / "rf_imbalance_per_class_metrics.csv")
print(OUTPUT_DIR / "rf_imbalance_summary.csv")
print(OUTPUT_DIR / "rf_imbalance_oof_predictions.csv")

print("\nIMPORTANT:")
print("Compare macro-F1 and per-class recall, not accuracy alone.")
print("These are cross-validation results, not independent test results.")

In [ ]:
from pathlib import Path

# Locate the project results directory
CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            p / "reports" / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

# Fix: use the existing variable `oof`
if "oof" not in globals():
    raise NameError(
        "The variable 'oof' does not exist. "
        "Rerun the cell that creates the long-format OOF predictions."
    )

# Save corrected OOF predictions
oof.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_predictions_long.csv",
    index=False,
)

# Save overall metrics
oof_summary.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_summary_recalculated.csv",
    index=False,
)

# Save per-class metrics
oof_class_metrics.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_per_class_metrics.csv",
    index=False,
)

# Save recall and F1 comparison tables
recall_table.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_recall_comparison.csv"
)

f1_table.to_csv(
    OUTPUT_DIR / "rf_imbalance_oof_f1_comparison.csv"
)

print("All analysis files saved successfully:")
print("1.", OUTPUT_DIR / "rf_imbalance_oof_predictions_long.csv")
print("2.", OUTPUT_DIR / "rf_imbalance_oof_summary_recalculated.csv")
print("3.", OUTPUT_DIR / "rf_imbalance_oof_per_class_metrics.csv")
print("4.", OUTPUT_DIR / "rf_imbalance_oof_recall_comparison.csv")
print("5.", OUTPUT_DIR / "rf_imbalance_oof_f1_comparison.csv")

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# LOAD SAVED RESULTS
# ============================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            p / "reports" / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

summary = pd.read_csv(
    OUTPUT_DIR / "rf_imbalance_oof_summary_recalculated.csv"
)

per_class = pd.read_csv(
    OUTPUT_DIR / "rf_imbalance_oof_per_class_metrics.csv"
)

recall = pd.read_csv(
    OUTPUT_DIR / "rf_imbalance_oof_recall_comparison.csv",
    index_col=0,
)

f1 = pd.read_csv(
    OUTPUT_DIR / "rf_imbalance_oof_f1_comparison.csv",
    index_col=0,
)

# ============================================================
# 1. OVERALL PERFORMANCE
# ============================================================

print("=" * 85)
print("OVERALL OOF PERFORMANCE")
print("=" * 85)

display(
    summary.sort_values(
        "oof_macro_f1",
        ascending=False
    ).round(4)
)

# ============================================================
# 2. CLASS-WISE RECALL
# ============================================================

print("\n" + "=" * 85)
print("OUT-OF-FOLD RECALL BY CLASS")
print("=" * 85)

display(recall.round(4))

# ============================================================
# 3. CLASS-WISE F1
# ============================================================

print("\n" + "=" * 85)
print("OUT-OF-FOLD F1-SCORE BY CLASS")
print("=" * 85)

display(f1.round(4))

# ============================================================
# 4. MINORITY CLASS COMPARISON
# ============================================================

minority_classes = [
    "Healthy",
    "URTI",
    "Bronchiectasis",
    "Pneumonia",
]

available_minority = [
    c for c in minority_classes
    if c in recall.index
]

print("\n" + "=" * 85)
print("MINORITY-CLASS RECALL")
print("=" * 85)

display(recall.loc[available_minority].round(4))

print("\nMean minority-class recall:")

display(
    recall.loc[available_minority]
    .mean(axis=0)
    .sort_values(ascending=False)
    .to_frame("mean_minority_recall")
    .round(4)
)

print("\n" + "=" * 85)
print("MINORITY-CLASS F1")
print("=" * 85)

display(f1.loc[available_minority].round(4))

print("\nMean minority-class F1:")

display(
    f1.loc[available_minority]
    .mean(axis=0)
    .sort_values(ascending=False)
    .to_frame("mean_minority_f1")
    .round(4)
)

# ============================================================
# 5. SAVE A COMPACT RESEARCH SUMMARY
# ============================================================

compact = summary[
    [
        "experiment",
        "oof_accuracy",
        "oof_balanced_accuracy",
        "oof_macro_f1",
        "oof_weighted_f1",
    ]
].copy()

compact.to_csv(
    OUTPUT_DIR / "rf_imbalance_final_comparison.csv",
    index=False,
)

print("\nFinal comparison saved to:")
print(OUTPUT_DIR / "rf_imbalance_final_comparison.csv")

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import ParameterGrid
from imblearn.ensemble import BalancedRandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# 1. LOCATE PROJECT AND LOAD DATA
# ============================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            p / "reports" / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

OUTPUT_DIR = (
    ROOT / "reports" / "research_audit"
    / "random_forest_optimization"
)

ASSIGNMENT_FILE = (
    OUTPUT_DIR / "five_class_grouped_cv_assignments.csv"
)

if not ASSIGNMENT_FILE.exists():
    raise FileNotFoundError(ASSIGNMENT_FILE)

df = pd.read_csv(ASSIGNMENT_FILE)

print("=" * 90)
print("BALANCED RANDOM FOREST HYPERPARAMETER OPTIMIZATION")
print("=" * 90)

print("Dataset shape:", df.shape)
print("Patients:", df["patient_id"].nunique())
print("Folds:", sorted(df["cv_fold"].unique()))

# ============================================================
# 2. DEFINE FEATURES AND LABELS
# ============================================================

excluded_columns = {
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
    "cv_fold",
}

feature_columns = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df["diagnosis"].astype(str).copy()
groups = df["patient_id"].copy()
folds = df["cv_fold"].astype(int).copy()

if X.isna().any().any():
    raise ValueError("Features contain missing values.")

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Features contain non-finite values.")

print("Features:", len(feature_columns))
print("Classes:", sorted(y.unique()))

# ============================================================
# 3. DEFINE COMPACT HYPERPARAMETER GRID
# ============================================================

param_grid = {
    "n_estimators": [300],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt"],
}

configurations = list(ParameterGrid(param_grid))

print("Configurations:", len(configurations))
print("Total model fits:", len(configurations) * len(folds.unique()))

# ============================================================
# 4. CROSS-VALIDATION LOOP
# ============================================================

fold_results = []
oof_results = []

for config_number, params in enumerate(configurations, start=1):

    print("\n" + "=" * 90)
    print(f"CONFIGURATION {config_number}/{len(configurations)}")
    print(params)
    print("=" * 90)

    start_time = time.time()

    predictions = pd.Series(
        index=df.index,
        dtype="object",
    )

    for fold in sorted(folds.unique()):

        train_idx = df.index[folds != fold]
        valid_idx = df.index[folds == fold]

        # Explicit leakage check
        train_patients = set(groups.loc[train_idx])
        valid_patients = set(groups.loc[valid_idx])

        overlap = train_patients.intersection(valid_patients)

        if overlap:
            raise RuntimeError(
                f"Patient leakage in fold {fold}: {overlap}"
            )

        model = BalancedRandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1,
            sampling_strategy="all",
            replacement=True,
            bootstrap=False,
        )

        model.fit(
            X.loc[train_idx],
            y.loc[train_idx],
        )

        pred = model.predict(X.loc[valid_idx]).astype(str)

        predictions.loc[valid_idx] = pred

        y_valid = y.loc[valid_idx]

        fold_results.append({
            "configuration": config_number,
            **params,
            "fold": fold,
            "accuracy": accuracy_score(y_valid, pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid, pred
            ),
            "macro_f1": f1_score(
                y_valid, pred, average="macro", zero_division=0
            ),
            "weighted_f1": f1_score(
                y_valid, pred, average="weighted", zero_division=0
            ),
        })

        print(
            f"Fold {fold}: "
            f"Accuracy={accuracy_score(y_valid, pred):.4f}, "
            f"BalancedAcc={balanced_accuracy_score(y_valid, pred):.4f}, "
            f"MacroF1={f1_score(y_valid, pred, average='macro', zero_division=0):.4f}"
        )

    if predictions.isna().any():
        raise RuntimeError(
            "Some rows did not receive out-of-fold predictions."
        )

    # Aggregate OOF metrics across all rows
    oof_accuracy = accuracy_score(y, predictions)
    oof_balanced_accuracy = balanced_accuracy_score(y, predictions)
    oof_macro_f1 = f1_score(
        y, predictions, average="macro", zero_division=0
    )
    oof_weighted_f1 = f1_score(
        y, predictions, average="weighted", zero_division=0
    )

    oof_results.append({
        "configuration": config_number,
        **params,
        "oof_accuracy": oof_accuracy,
        "oof_balanced_accuracy": oof_balanced_accuracy,
        "oof_macro_f1": oof_macro_f1,
        "oof_weighted_f1": oof_weighted_f1,
        "mean_fold_macro_f1": np.mean([
            row["macro_f1"]
            for row in fold_results
            if row["configuration"] == config_number
        ]),
        "std_fold_macro_f1": np.std([
            row["macro_f1"]
            for row in fold_results
            if row["configuration"] == config_number
        ], ddof=1),
        "runtime_seconds": time.time() - start_time,
    })

    # Store OOF predictions for each configuration
    for idx in df.index:
        oof_results.append({
            "configuration": config_number,
            "patient_id": groups.loc[idx],
            "diagnosis": y.loc[idx],
            "cv_fold": folds.loc[idx],
            "prediction": predictions.loc[idx],
        })

    print(
        f"OOF Macro-F1={oof_macro_f1:.4f} | "
        f"OOF Balanced Accuracy={oof_balanced_accuracy:.4f} | "
        f"Runtime={time.time() - start_time:.1f}s"
    )

# ============================================================
# 5. SAVE AND DISPLAY RESULTS
# ============================================================

fold_results_df = pd.DataFrame(fold_results)
summary_df = pd.DataFrame(oof_results[:0])

# Reconstruct the configuration summary rows from fold results
summary_df = pd.DataFrame([
    {
        "configuration": row["configuration"],
        **{
            key: row[key]
            for key in [
                "n_estimators",
                "max_depth",
                "min_samples_leaf",
                "max_features",
            ]
        },
        "oof_accuracy": row["oof_accuracy"],
        "oof_balanced_accuracy": row["oof_balanced_accuracy"],
        "oof_macro_f1": row["oof_macro_f1"],
        "oof_weighted_f1": row["oof_weighted_f1"],
        "mean_fold_macro_f1": row["mean_fold_macro_f1"],
        "std_fold_macro_f1": row["std_fold_macro_f1"],
        "runtime_seconds": row["runtime_seconds"],
    }
    for row in oof_results
    if "oof_macro_f1" in row
])

# The configuration summary is already available from the loop;
# rebuild it from the per-configuration metric records.
summary_rows = []

for config_number, params in enumerate(configurations, start=1):

    matching_folds = fold_results_df[
        fold_results_df["configuration"] == config_number
    ]

    # Recalculate aggregate OOF metrics from saved predictions
    # in the in-memory OOF record list.
    config_predictions = pd.DataFrame([
        row for row in oof_results
        if row["configuration"] == config_number
        and "prediction" in row
    ])

    config_predictions = config_predictions.sort_values(
        ["patient_id", "diagnosis", "cv_fold"]
    )

    # Fold means are included as descriptive metrics.
    summary_rows.append({
        "configuration": config_number,
        **params,
        "mean_fold_accuracy": matching_folds["accuracy"].mean(),
        "mean_fold_balanced_accuracy": (
            matching_folds["balanced_accuracy"].mean()
        ),
        "mean_fold_macro_f1": matching_folds["macro_f1"].mean(),
        "std_fold_macro_f1": matching_folds["macro_f1"].std(),
        "mean_fold_weighted_f1": matching_folds["weighted_f1"].mean(),
    })

fold_summary_df = pd.DataFrame(summary_rows)

# Calculate aggregate OOF scores from the saved predictions
for col in [
    "oof_accuracy",
    "oof_balanced_accuracy",
    "oof_macro_f1",
    "oof_weighted_f1",
]:
    fold_summary_df[col] = np.nan

for config_number in range(1, len(configurations) + 1):

    config_predictions = [
        row for row in oof_results
        if row["configuration"] == config_number
        and "prediction" in row
    ]

    y_true = [row["diagnosis"] for row in config_predictions]
    y_pred = [row["prediction"] for row in config_predictions]

    mask = np.asarray(y_true) != ""
    y_true = np.asarray(y_true)[mask]
    y_pred = np.asarray(y_pred)[mask]

    row_idx = fold_summary_df.index[
        fold_summary_df["configuration"] == config_number
    ][0]

    fold_summary_df.loc[row_idx, "oof_accuracy"] = accuracy_score(
        y_true, y_pred
    )
    fold_summary_df.loc[row_idx, "oof_balanced_accuracy"] = (
        balanced_accuracy_score(y_true, y_pred)
    )
    fold_summary_df.loc[row_idx, "oof_macro_f1"] = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    fold_summary_df.loc[row_idx, "oof_weighted_f1"] = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

fold_summary_df = fold_summary_df.sort_values(
    ["oof_macro_f1", "oof_balanced_accuracy"],
    ascending=False,
)

fold_results_df.to_csv(
    OUTPUT_DIR / "brf_tuning_fold_metrics.csv",
    index=False,
)

fold_summary_df.to_csv(
    OUTPUT_DIR / "brf_tuning_summary.csv",
    index=False,
)

pd.DataFrame(oof_results).to_csv(
    OUTPUT_DIR / "brf_tuning_oof_predictions.csv",
    index=False,
)

print("\n" + "=" * 90)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 90)

display(fold_summary_df.round(4))

print("\nSaved files:")
print(OUTPUT_DIR / "brf_tuning_fold_metrics.csv")
print(OUTPUT_DIR / "brf_tuning_summary.csv")
print(OUTPUT_DIR / "brf_tuning_oof_predictions.csv")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

summary_path = RESULTS_DIR / "brf_tuning_summary.csv"

summary = pd.read_csv(summary_path)

print("=" * 100)
print("BALANCED RANDOM FOREST TUNING SUMMARY")
print("=" * 100)

print("\nColumns:")
print(summary.columns.tolist())

print("\nResults:")
print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nBest configuration by OOF Macro-F1:")
print(
    summary.loc[
        summary["oof_macro_f1"].idxmax()
    ].to_string()
)

print("\nBest configuration by OOF Balanced Accuracy:")
print(
    summary.loc[
        summary["oof_balanced_accuracy"].idxmax()
    ].to_string()
)

In [ ]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1. LOCATE THE REPOSITORY ROOT
# ---------------------------------------------------------

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (parent / "reports" / "research_audit").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root automatically.\n"
        f"Current directory: {CURRENT_DIR}\n"
        "Set ROOT manually to your repository directory."
    )

print("Repository root:", ROOT)

# ---------------------------------------------------------
# 2. LOCATE THE TUNING SUMMARY
# ---------------------------------------------------------

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

summary_path = RESULTS_DIR / "brf_tuning_summary.csv"

if not summary_path.exists():
    print("\nSummary file not found at the expected location.")
    print("Searching the repository for the summary file...")

    matches = list(ROOT.rglob("brf_tuning_summary.csv"))

    if not matches:
        raise FileNotFoundError(
            "brf_tuning_summary.csv was not found in the repository.\n"
            "Check whether the tuning notebook successfully saved the file."
        )

    summary_path = matches[0]

# ---------------------------------------------------------
# 3. LOAD AND DISPLAY THE RESULTS
# ---------------------------------------------------------

summary = pd.read_csv(summary_path)

print("\nSummary file:", summary_path)
print("\nColumns:")
print(summary.columns.tolist())

print("\n" + "=" * 100)
print("BALANCED RANDOM FOREST HYPERPARAMETER TUNING RESULTS")
print("=" * 100)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ---------------------------------------------------------
# 4. IDENTIFY THE BEST CONFIGURATIONS
# ---------------------------------------------------------

required_columns = {
    "oof_macro_f1",
    "oof_balanced_accuracy"
}

missing_columns = required_columns - set(summary.columns)

if missing_columns:
    raise ValueError(
        f"Missing expected metric columns: {sorted(missing_columns)}\n"
        f"Available columns: {summary.columns.tolist()}"
    )

print("\n" + "=" * 100)
print("BEST CONFIGURATION BY OOF MACRO-F1")
print("=" * 100)

print(
    summary.loc[
        summary["oof_macro_f1"].idxmax()
    ].to_string()
)

print("\n" + "=" * 100)
print("BEST CONFIGURATION BY OOF BALANCED ACCURACY")
print("=" * 100)

print(
    summary.loc[
        summary["oof_balanced_accuracy"].idxmax()
    ].to_string()
)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

# =========================================================
# 1. LOCATE PROJECT AND LOAD OOF PREDICTIONS
# =========================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            parent
            / "reports"
            / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

PREDICTIONS_PATH = RESULTS_DIR / "brf_tuning_oof_predictions.csv"

oof = pd.read_csv(PREDICTIONS_PATH)

# =========================================================
# 2. REMOVE SUMMARY ROWS
# =========================================================

required_columns = [
    "configuration",
    "patient_id",
    "diagnosis",
    "cv_fold",
    "prediction",
]

oof = oof.dropna(subset=required_columns).copy()

oof["configuration"] = pd.to_numeric(
    oof["configuration"], errors="coerce"
)

oof["cv_fold"] = pd.to_numeric(
    oof["cv_fold"], errors="coerce"
)

oof = oof.dropna(
    subset=["configuration", "cv_fold"]
).copy()

oof["configuration"] = oof["configuration"].astype(int)
oof["cv_fold"] = oof["cv_fold"].astype(int)

oof["diagnosis"] = oof["diagnosis"].astype(str).str.strip()
oof["prediction"] = oof["prediction"].astype(str).str.strip()

print("Cleaned OOF prediction shape:", oof.shape)

# =========================================================
# 3. DEFINE CLASSES AND CONFIGURATIONS
# =========================================================

class_labels = [
    "Bronchiectasis",
    "COPD",
    "Healthy",
    "Pneumonia",
    "URTI",
]

selected_configurations = [1, 3]

# =========================================================
# 4. VALIDATE PREDICTION COUNTS
# =========================================================

print("\n" + "=" * 80)
print("OOF PREDICTION VALIDATION")
print("=" * 80)

for config_id in selected_configurations:
    subset = oof[oof["configuration"] == config_id]

    print(f"\nConfiguration {config_id}")
    print("Predictions:", len(subset))
    print("Unique patients:", subset["patient_id"].nunique())
    print("Fold counts:")
    print(subset["cv_fold"].value_counts().sort_index())

    if len(subset) != 6700:
        print(
            f"WARNING: Expected 6700 predictions, "
            f"but found {len(subset)}."
        )

    if subset.empty:
        raise ValueError(
            f"No predictions found for configuration {config_id}."
        )

# =========================================================
# 5. EVALUATE CONFIGURATIONS
# =========================================================

overall_results = []
per_class_results = []

for config_id in selected_configurations:

    subset = oof[
        oof["configuration"] == config_id
    ].copy()

    y_true = subset["diagnosis"]
    y_pred = subset["prediction"]

    if not set(y_true.unique()).issubset(set(class_labels)):
        raise ValueError("Unexpected labels found in diagnosis.")

    if not set(y_pred.unique()).issubset(set(class_labels)):
        raise ValueError("Unexpected labels found in predictions.")

    print("\n" + "=" * 80)
    print(f"CONFIGURATION {config_id}")
    print("=" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=class_labels,
            target_names=class_labels,
            digits=4,
            zero_division=0,
        )
    )

    overall_results.append(
        {
            "configuration": config_id,
            "n_predictions": len(subset),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "macro_f1": f1_score(
                y_true,
                y_pred,
                labels=class_labels,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_true,
                y_pred,
                labels=class_labels,
                average="weighted",
                zero_division=0,
            ),
        }
    )

    report = classification_report(
        y_true,
        y_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )

    for label in class_labels:
        per_class_results.append(
            {
                "configuration": config_id,
                "class": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1_score": report[label]["f1-score"],
                "support": report[label]["support"],
            }
        )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_labels,
    )

    print("\nConfusion Matrix (rows=true, columns=predicted):")

    print(
        pd.DataFrame(
            cm,
            index=class_labels,
            columns=class_labels,
        )
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_labels,
    )

    fig, ax = plt.subplots(figsize=(9, 7))

    display.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        xticks_rotation=45,
        colorbar=False,
    )

    ax.set_title(
        f"Balanced Random Forest - Configuration {config_id}"
    )

    plt.tight_layout()
    plt.show()

# =========================================================
# 6. SAVE COMPARISON RESULTS
# =========================================================

overall_df = pd.DataFrame(overall_results)
per_class_df = pd.DataFrame(per_class_results)

overall_path = (
    RESULTS_DIR / "brf_selected_configurations_comparison.csv"
)

per_class_path = (
    RESULTS_DIR / "brf_selected_configurations_per_class.csv"
)

overall_df.to_csv(overall_path, index=False)
per_class_df.to_csv(per_class_path, index=False)

print("\n" + "=" * 80)
print("OVERALL CONFIGURATION COMPARISON")
print("=" * 80)

print(overall_df.to_string(index=False))

print("\n" + "=" * 80)
print("PER-CLASS METRICS")
print("=" * 80)

print(
    per_class_df.pivot(
        index="class",
        columns="configuration",
        values=["precision", "recall", "f1_score"],
    ).round(4).to_string()
)

print("\nSaved files:")
print(overall_path)
print(per_class_path)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)

# =========================================================
# 1. LOCATE PROJECT AND LOAD OOF PREDICTIONS
# =========================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            parent
            / "reports"
            / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

PREDICTIONS_PATH = RESULTS_DIR / "brf_tuning_oof_predictions.csv"

oof = pd.read_csv(PREDICTIONS_PATH)

# Remove summary rows; retain only actual OOF predictions.
oof = oof.dropna(
    subset=[
        "configuration",
        "patient_id",
        "diagnosis",
        "cv_fold",
        "prediction",
    ]
).copy()

oof["configuration"] = oof["configuration"].astype(int)
oof["cv_fold"] = oof["cv_fold"].astype(int)
oof["diagnosis"] = oof["diagnosis"].astype(str).str.strip()
oof["prediction"] = oof["prediction"].astype(str).str.strip()

CLASS_LABELS = [
    "Bronchiectasis",
    "COPD",
    "Healthy",
    "Pneumonia",
    "URTI",
]

CONFIGURATIONS = [1, 3]

# =========================================================
# 2. CALCULATE FOLD-WISE METRICS
# =========================================================

fold_results = []
oof_results = []

for config_id in CONFIGURATIONS:

    config_df = oof.loc[
        oof["configuration"] == config_id
    ].copy()

    if len(config_df) != 6700:
        raise ValueError(
            f"Configuration {config_id}: expected 6700 OOF "
            f"predictions, found {len(config_df)}."
        )

    # Calculate metrics separately for every fold.
    for fold in sorted(config_df["cv_fold"].unique()):

        fold_df = config_df.loc[
            config_df["cv_fold"] == fold
        ]

        y_true = fold_df["diagnosis"]
        y_pred = fold_df["prediction"]

        fold_results.append(
            {
                "configuration": config_id,
                "cv_fold": fold,
                "n_samples": len(fold_df),
                "accuracy": accuracy_score(y_true, y_pred),
                "balanced_accuracy": balanced_accuracy_score(
                    y_true, y_pred
                ),
                "macro_f1": f1_score(
                    y_true,
                    y_pred,
                    labels=CLASS_LABELS,
                    average="macro",
                    zero_division=0,
                ),
                "weighted_f1": f1_score(
                    y_true,
                    y_pred,
                    labels=CLASS_LABELS,
                    average="weighted",
                    zero_division=0,
                ),
            }
        )

    # Calculate metrics across all out-of-fold predictions.
    y_true = config_df["diagnosis"]
    y_pred = config_df["prediction"]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_LABELS,
        zero_division=0,
    )

    oof_results.append(
        {
            "configuration": config_id,
            "oof_accuracy": accuracy_score(y_true, y_pred),
            "oof_balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "oof_macro_f1": f1_score(
                y_true,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            ),
            "oof_weighted_f1": f1_score(
                y_true,
                y_pred,
                labels=CLASS_LABELS,
                average="weighted",
                zero_division=0,
            ),
        }
    )

# =========================================================
# 3. SUMMARIZE FOLD-WISE PERFORMANCE
# =========================================================

fold_df = pd.DataFrame(fold_results)
oof_summary_df = pd.DataFrame(oof_results)

fold_summary = (
    fold_df.groupby("configuration")
    .agg(
        mean_fold_accuracy=("accuracy", "mean"),
        std_fold_accuracy=("accuracy", "std"),
        mean_fold_balanced_accuracy=("balanced_accuracy", "mean"),
        std_fold_balanced_accuracy=("balanced_accuracy", "std"),
        mean_fold_macro_f1=("macro_f1", "mean"),
        std_fold_macro_f1=("macro_f1", "std"),
        mean_fold_weighted_f1=("weighted_f1", "mean"),
        std_fold_weighted_f1=("weighted_f1", "std"),
    )
    .reset_index()
)

comparison = fold_summary.merge(
    oof_summary_df,
    on="configuration",
    how="inner",
)

print("\n" + "=" * 100)
print("FOLD-WISE PERFORMANCE")
print("=" * 100)

print(
    fold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print("\n" + "=" * 100)
print("OVERALL CONFIGURATION COMPARISON")
print("=" * 100)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

# =========================================================
# 4. COMPARE PER-CLASS OOF PERFORMANCE
# =========================================================

class_results = []

for config_id in CONFIGURATIONS:

    config_df = oof.loc[
        oof["configuration"] == config_id
    ]

    precision, recall, f1, support = precision_recall_fscore_support(
        config_df["diagnosis"],
        config_df["prediction"],
        labels=CLASS_LABELS,
        zero_division=0,
    )

    for i, class_name in enumerate(CLASS_LABELS):
        class_results.append(
            {
                "configuration": config_id,
                "class": class_name,
                "precision": precision[i],
                "recall": recall[i],
                "f1_score": f1[i],
                "support": support[i],
            }
        )

class_df = pd.DataFrame(class_results)

print("\n" + "=" * 100)
print("PER-CLASS PERFORMANCE COMPARISON")
print("=" * 100)

print(
    class_df.pivot(
        index="class",
        columns="configuration",
        values=["precision", "recall", "f1_score"],
    ).round(4).to_string()
)

# =========================================================
# 5. SAVE RESULTS
# =========================================================

fold_path = RESULTS_DIR / "brf_selected_fold_metrics.csv"
summary_path = RESULTS_DIR / "brf_selected_fold_summary.csv"
class_path = RESULTS_DIR / "brf_selected_per_class_metrics.csv"

fold_df.to_csv(fold_path, index=False)
comparison.to_csv(summary_path, index=False)
class_df.to_csv(class_path, index=False)

print("\nSaved files:")
print(fold_path)
print(summary_path)
print(class_path)

# =========================================================
# 6. AUTOMATICALLY IDENTIFY BEST CONFIGURATIONS
# =========================================================

best_macro_f1 = comparison.loc[
    comparison["oof_macro_f1"].idxmax()
]

best_balanced_accuracy = comparison.loc[
    comparison["oof_balanced_accuracy"].idxmax()
]

print("\n" + "=" * 100)
print("BEST CONFIGURATION BY OOF MACRO-F1")
print("=" * 100)

print(best_macro_f1.to_string())

print("\n" + "=" * 100)
print("BEST CONFIGURATION BY OOF BALANCED ACCURACY")
print("=" * 100)

print(best_balanced_accuracy.to_string())

print("\nEvaluation completed successfully.")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon

# =========================================================
# 1. LOCATE PROJECT AND LOAD FOLD RESULTS
# =========================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            parent
            / "reports"
            / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

FOLD_PATH = RESULTS_DIR / "brf_selected_fold_metrics.csv"
SUMMARY_PATH = RESULTS_DIR / "brf_selected_fold_summary.csv"
CLASS_PATH = RESULTS_DIR / "brf_selected_per_class_metrics.csv"

fold_df = pd.read_csv(FOLD_PATH)
summary_df = pd.read_csv(SUMMARY_PATH)
class_df = pd.read_csv(CLASS_PATH)

# =========================================================
# 2. COMPARE FOLD-WISE PERFORMANCE
# =========================================================

metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]

print("=" * 90)
print("CONFIGURATION 1 VS CONFIGURATION 3")
print("=" * 90)

for metric in metrics:

    pivot = fold_df.pivot(
        index="cv_fold",
        columns="configuration",
        values=metric,
    )

    pivot.columns = [
        f"Configuration_{int(column)}"
        for column in pivot.columns
    ]

    print(f"\n{metric.upper().replace('_', ' ')}")
    print(pivot.round(4).to_string())

# =========================================================
# 3. PLOT FOLD-WISE MACRO-F1 AND BALANCED ACCURACY
# =========================================================

plot_metrics = [
    ("macro_f1", "Macro-F1 by Fold"),
    ("balanced_accuracy", "Balanced Accuracy by Fold"),
]

for metric, title in plot_metrics:

    pivot = fold_df.pivot(
        index="cv_fold",
        columns="configuration",
        values=metric,
    )

    ax = pivot.plot(
        kind="bar",
        figsize=(9, 5),
    )

    ax.set_title(title)
    ax.set_xlabel("Cross-validation fold")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_xticklabels(
        [f"Fold {x}" for x in pivot.index],
        rotation=0,
    )
    ax.legend(
        ["Configuration 1", "Configuration 3"],
        title="Model",
    )

    plt.tight_layout()
    plt.show()

# =========================================================
# 4. CALCULATE PER-CLASS RECALL IMPROVEMENTS
# =========================================================

recall_pivot = class_df.pivot(
    index="class",
    columns="configuration",
    values="recall",
)

recall_pivot.columns = [
    f"Configuration_{int(column)}"
    for column in recall_pivot.columns
]

recall_pivot["Recall_difference_3_minus_1"] = (
    recall_pivot["Configuration_3"]
    - recall_pivot["Configuration_1"]
)

print("\n" + "=" * 90)
print("PER-CLASS RECALL COMPARISON")
print("=" * 90)

print(recall_pivot.round(4).to_string())

# =========================================================
# 5. COMPARE MACRO-F1 AND BALANCED ACCURACY
# =========================================================

metric_comparison = summary_df[
    [
        "configuration",
        "oof_accuracy",
        "oof_balanced_accuracy",
        "oof_macro_f1",
        "oof_weighted_f1",
    ]
].copy()

metric_comparison["configuration"] = (
    metric_comparison["configuration"].astype(int)
)

print("\n" + "=" * 90)
print("OVERALL OOF METRIC COMPARISON")
print("=" * 90)

print(
    metric_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

# =========================================================
# 6. SAVE COMPARISON ARTIFACTS
# =========================================================

recall_comparison_path = (
    RESULTS_DIR / "brf_configuration_recall_comparison.csv"
)

recall_pivot.to_csv(recall_comparison_path)

print("\nSaved:")
print(recall_comparison_path)

# =========================================================
# 7. PROVISIONAL MODEL SELECTION
# =========================================================

best_macro = metric_comparison.loc[
    metric_comparison["oof_macro_f1"].idxmax()
]

best_balanced = metric_comparison.loc[
    metric_comparison["oof_balanced_accuracy"].idxmax()
]

print("\n" + "=" * 90)
print("PROVISIONAL SELECTION")
print("=" * 90)

print(
    f"Highest OOF Macro-F1: Configuration "
    f"{int(best_macro['configuration'])} "
    f"({best_macro['oof_macro_f1']:.4f})"
)

print(
    f"Highest OOF Balanced Accuracy: Configuration "
    f"{int(best_balanced['configuration'])} "
    f"({best_balanced['oof_balanced_accuracy']:.4f})"
)

print(
    "\nInterpretation: Configuration 1 is the current candidate "
    "when macro-F1 is the primary selection metric. "
    "Configuration 3 is the candidate when balanced accuracy "
    "is the primary selection metric."
)

print(
    "\nNote: Five folds provide limited evidence for a formal "
    "statistical significance test. Do not interpret small "
    "fold-wise differences as statistically significant."
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 1. LOCATE PROJECT AND LOAD RESULTS
# =========================================================

CURRENT_DIR = Path.cwd()

ROOT = next(
    (
        parent
        for parent in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (
            parent
            / "reports"
            / "research_audit"
            / "random_forest_optimization"
        ).exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

RESULTS_DIR = (
    ROOT
    / "reports"
    / "research_audit"
    / "random_forest_optimization"
)

OOF_PATH = RESULTS_DIR / "brf_tuning_oof_predictions.csv"

oof = pd.read_csv(OOF_PATH)

oof = oof.dropna(
    subset=[
        "configuration",
        "patient_id",
        "diagnosis",
        "cv_fold",
        "prediction",
    ]
).copy()

oof["configuration"] = oof["configuration"].astype(int)
oof["cv_fold"] = oof["cv_fold"].astype(int)

CLASS_LABELS = [
    "Bronchiectasis",
    "COPD",
    "Healthy",
    "Pneumonia",
    "URTI",
]

# =========================================================
# 2. ANALYZE PATIENT-LEVEL PERFORMANCE
# =========================================================

# Cycle-level metrics can be dominated by patients with many
# recordings. Report patient-level results as an additional
# robustness analysis, not as a replacement for the primary
# cycle-level evaluation.

patient_results = []

for config_id in [1, 3]:

    config_df = oof.loc[
        oof["configuration"] == config_id
    ].copy()

    for patient_id, patient_df in config_df.groupby("patient_id"):

        y_true = patient_df["diagnosis"].to_numpy()
        y_pred = patient_df["prediction"].to_numpy()

        # Only calculate per-patient accuracy here. Macro-F1
        # requires sufficient class support per patient.
        patient_results.append(
            {
                "configuration": config_id,
                "patient_id": patient_id,
                "n_cycles": len(patient_df),
                "accuracy": np.mean(y_true == y_pred),
            }
        )

patient_df = pd.DataFrame(patient_results)

patient_summary = (
    patient_df.groupby("configuration")
    .agg(
        mean_patient_accuracy=("accuracy", "mean"),
        std_patient_accuracy=("accuracy", "std"),
        median_patient_accuracy=("accuracy", "median"),
        number_of_patients=("patient_id", "nunique"),
    )
    .reset_index()
)

print("=" * 90)
print("PATIENT-LEVEL ACCURACY ROBUSTNESS ANALYSIS")
print("=" * 90)

print(patient_summary.round(4).to_string(index=False))

patient_df.to_csv(
    RESULTS_DIR / "brf_patient_level_accuracy.csv",
    index=False,
)

patient_summary.to_csv(
    RESULTS_DIR / "brf_patient_level_summary.csv",
    index=False,
)

# =========================================================
# 3. PLOT PER-CLASS RECALL COMPARISON
# =========================================================

from sklearn.metrics import recall_score

recall_results = []

for config_id in [1, 3]:

    config_df = oof.loc[
        oof["configuration"] == config_id
    ]

    recalls = recall_score(
        config_df["diagnosis"],
        config_df["prediction"],
        labels=CLASS_LABELS,
        average=None,
        zero_division=0,
    )

    for label, recall in zip(CLASS_LABELS, recalls):
        recall_results.append(
            {
                "configuration": config_id,
                "class": label,
                "recall": recall,
            }
        )

recall_df = pd.DataFrame(recall_results)

recall_pivot = recall_df.pivot(
    index="class",
    columns="configuration",
    values="recall",
).reindex(CLASS_LABELS)

ax = recall_pivot.plot(
    kind="bar",
    figsize=(10, 6),
)

ax.set_title("Per-Class Recall: Balanced Random Forest")
ax.set_xlabel("Respiratory diagnosis")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.legend(
    ["Configuration 1: min_samples_leaf=1",
     "Configuration 3: min_samples_leaf=5"],
    title="Configuration",
)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

recall_df.to_csv(
    RESULTS_DIR / "brf_per_class_recall_comparison.csv",
    index=False,
)

# =========================================================
# 4. FINAL PROVISIONAL MODEL DECISION
# =========================================================

print("\n" + "=" * 90)
print("PROVISIONAL RESEARCH DECISION")
print("=" * 90)

print(
    """
Configuration 1:
    max_depth=None
    max_features='sqrt'
    min_samples_leaf=1
    n_estimators=300

    OOF Macro-F1:           0.4110
    OOF Balanced Accuracy:  0.5198
    OOF Accuracy:           0.7743

Configuration 3:
    max_depth=None
    max_features='sqrt'
    min_samples_leaf=5
    n_estimators=300

    OOF Macro-F1:           0.3976
    OOF Balanced Accuracy:  0.5478
    OOF Accuracy:           0.7294

Decision:
    Keep Configuration 1 as the provisional primary model
    when macro-F1 is the primary metric.

    Retain Configuration 3 as a comparison model because
    it has better balanced accuracy and Bronchiectasis recall.

    Do not claim statistical superiority from these results
    alone. Both configurations were evaluated on the same
    five patient-grouped folds.
    """
)

print("Results saved to:", RESULTS_DIR)